# Project 4: Multi-Agent RL Baseline

Train multiple agents to cooperate in the Simple Spread environment.

**Runtime:** a few minutes for the baseline

## Setup

In [ ]:
# MPE moved out of PettingZoo into its own package (mpe2).
!pip install stable-baselines3 mpe2 supersuit -q

In [ ]:
from mpe2 import simple_spread_v3
import supersuit as ss
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy

## Create Multi-Agent Environment

**Simple Spread:**
- 3 agents (cooperative)
- 3 landmarks
- Goal: Each agent covers a different landmark
- Reward: shared by all agents - the further each landmark is from its nearest agent,
  the more negative - plus a penalty for agents colliding with each other

In [ ]:
# Create base environment
env = simple_spread_v3.parallel_env(
    N=3,                    # Number of agents
    max_cycles=25,          # Episode length
    continuous_actions=False
)

# PettingZoo only populates env.agents when the episode starts, so reset()
# has to come BEFORE asking how many agents there are - otherwise you get
# AttributeError: 'aec_to_parallel_wrapper' object has no attribute 'agents'
observations, infos = env.reset(seed=42)

print(f"Environment: Simple Spread")
print(f"  Number of agents: {env.num_agents}")
print(f"  Agents: {env.agents}")

agent = env.agents[0]

print(f"\nAgent '{agent}':")
print(f"  Observation shape: {observations[agent].shape}")
print(f"  Action space: {env.action_space(agent)}")
print(f"    (5 actions: no-op, left, right, up, down)")

## Preprocess Environment

Use SuperSuit to make the multi-agent env usable by SB3. Each agent becomes one slot
of a vectorised environment - so SB3 sees "3 environments", and **one PPO policy is
trained on the experience of all three agents** (parameter sharing). Each agent still
acts on its own observation.

In [ ]:
# Wrap environment for SB3
env = simple_spread_v3.parallel_env(N=3, max_cycles=25, continuous_actions=False)

# Each agent becomes one slot of a vectorised env -> one shared policy for all agents
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')

print("Environment wrapped for SB3")
print(f"  Observation space: {env.observation_space}")
print(f"  Action space: {env.action_space}")

## Train Agents (PPO with a Shared Policy)

All agents use the same network, trained on all of their experience. Giving each
agent its own policy instead (independent learners) is one of the extensions in the
README.

In [ ]:
# Create PPO agent
model = PPO(
    MlpPolicy,
    env,
    learning_rate=1e-3,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    # NOTE: no `seed=` here. SB3 would call env.seed(), which supersuit's
    # ConcatVecEnv does not implement - so this run is not seeded and results
    # will differ from run to run.
)

print("\nTraining agents (~5 minutes)...\n")
model.learn(total_timesteps=100_000)
print("\nTraining complete!")

## Evaluate Trained Agents

In [ ]:
# Evaluate
def evaluate_multi_agent(model, n_episodes=20, random_actions=False):
    """Evaluate the policy - or, with random_actions=True, a random baseline."""
    episode_rewards = []
    
    for episode in range(n_episodes):
        obs = env.reset()
        done = False
        episode_reward = 0.0
        
        # `done` and `reward` are arrays with one entry per agent, so they have
        # to be reduced to a scalar - `while not done:` raises a ValueError.
        while not done:
            if random_actions:
                action = np.array([env.action_space.sample() for _ in range(env.num_envs)])
            else:
                action, _ = model.predict(obs, deterministic=True)
            obs, reward, dones, info = env.step(action)
            episode_reward += float(np.sum(reward))   # total across all agents
            done = bool(np.all(dones))
        
        episode_rewards.append(episode_reward)
    
    return episode_rewards


rewards = evaluate_multi_agent(model, n_episodes=20)
random_rewards = evaluate_multi_agent(model, n_episodes=20, random_actions=True)

print(f"\n{'='*50}")
print(f"Evaluation (20 episodes, reward summed over agents):")
print(f"{'='*50}")
print(f"  Trained policy: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")
print(f"  Random actions: {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}")
print(f"{'='*50}")

# Higher (less negative) is better. There is no fixed "solved" score here, so
# compare against random behaviour.
if np.mean(rewards) > np.mean(random_rewards) + np.std(random_rewards):
    print(f"\n✓ Clearly better than random - the agents have learned something.")
else:
    print(f"\n⚠ Not clearly better than random yet. Coordination takes longer to learn")
    print(f"  than 100k steps - training longer is the first thing to try.")

## Visualize Agent Behavior

In [ ]:
# Record an episode with the trained policy
raw_env = simple_spread_v3.parallel_env(
    N=3, max_cycles=25, 
    continuous_actions=False,
    render_mode='rgb_array'
)

observations, infos = raw_env.reset(seed=42)
frames = []
episode_reward = {agent: 0 for agent in raw_env.agents}

for step in range(25):  # Max steps
    # Get frame
    frames.append(raw_env.render())
    
    # Each agent acts on its own observation, with the shared trained policy
    actions = {agent: int(model.predict(observations[agent], deterministic=True)[0])
               for agent in raw_env.agents}
    
    observations, rewards_step, terminations, truncations, infos = raw_env.step(actions)
    
    for agent in raw_env.agents:
        episode_reward[agent] += rewards_step[agent]
    
    if all(terminations.values()) or all(truncations.values()):
        break

print(f"Episode rewards per agent:")
for agent, reward in episode_reward.items():
    print(f"  {agent}: {reward:.2f}")

# Show sample frames
if len(frames) >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    indices = [0, len(frames)//2, -1]
    
    for i, idx in enumerate(indices):
        axes[i].imshow(frames[idx])
        axes[i].set_title(f"Step {idx if idx >= 0 else len(frames)+idx}")
        axes[i].axis('off')
    
    plt.suptitle("Trained Agents (blue = agents, dark grey = landmarks)")
    plt.tight_layout()
    plt.show()

print("\nNote: blue circles are agents, dark grey circles are landmarks")
print("Goal: Each agent should cover a different landmark")

## Analyze Coordination

The reward mixes coverage and collisions. A more direct measure of coverage: for each
landmark, the distance to its **nearest** agent, averaged over landmarks. Lower is
better, and it is close to 0 only when every landmark has an agent on it.

In [ ]:
def coverage_distance(env_test):
    """Mean distance from each landmark to its nearest agent."""
    world = env_test.unwrapped.world
    return np.mean([min(np.linalg.norm(a.state.p_pos - lm.state.p_pos) for a in world.agents)
                    for lm in world.landmarks])


def analyze_coordination(model=None, n_episodes=20):
    """Coverage distance at the end of each episode (model=None: random actions)."""
    env_test = simple_spread_v3.parallel_env(N=3, max_cycles=25, continuous_actions=False)
    final_distances = []
    
    for episode in range(n_episodes):
        observations, _ = env_test.reset(seed=episode)
        
        for step in range(25):
            if model is None:
                actions = {agent: env_test.action_space(agent).sample() for agent in env_test.agents}
            else:
                actions = {agent: int(model.predict(observations[agent], deterministic=True)[0])
                           for agent in env_test.agents}
            last_distance = coverage_distance(env_test)
            observations, _, terminations, truncations, _ = env_test.step(actions)
            
            if all(terminations.values()) or all(truncations.values()):
                break
        
        final_distances.append(last_distance)
    
    return final_distances


trained_cov = analyze_coordination(model)
random_cov = analyze_coordination(None)

print("Coverage distance at the end of the episode (lower = better spread):")
print(f"  Trained policy: {np.mean(trained_cov):.2f} ± {np.std(trained_cov):.2f}")
print(f"  Random actions: {np.mean(random_cov):.2f} ± {np.std(random_cov):.2f}")

## Next Steps

Improve multi-agent coordination:

1. **Independent policies:** Give each agent its own policy and compare with the shared one
2. **Communication:** Add message passing between agents
3. **Curriculum:** Start with 2 agents, increase to 3
4. **Competitive:** Try Simple Tag (predator-prey)
5. **Different algorithms:** MADDPG, QMIX, CommNet

See `project4_multi_agent_README.md` for detailed ideas!

**Note:** Multi-agent RL is an active research area with many open challenges!